# New-RscMssqlExport
In Rubrik, an export is a database recovery that you control all the parameters of. You can restore a database to a different instance or to the same instance as a different name. You can control the paths and filenames of the recovered database. You can control if the database should be fully recovered or left in NORECOVERY mode. You can control if you overwrite the target database or not. 

New-RscMssqlExport has multiple parameters, but the important ones to understand for most workflows are:
- RscMssqlDatabase: This will be the object returned from Get-RscMssqlDatabase
- RecoveryDateTime: This will be a value returned from Get-RscMssqlDatabaseRecoveryPoint
- TargetMssqlInstance: This will be the object returned from Get-RscMssqlInstance
- TargetDatabaseName: The name of the newly created database from the restore
- TargetDataPath & TargeLogPath: Used for the location of all data and log files of the database. This would be for the Simple Method in the UI.
- TargetFilePaths: Used for the location of all data and log files of the database. This would be the Advanced Method in the UI.  You are providing a hashtable with each file listed. You can get a list of files from Get-RscMssqlDatabaseFiles
- Overwrite: Used to denote if the target database should be overwritten 
- FinishRecovery: Ommitting this switch will restore the database into NORECOVERY mode. 

# Workflow
To start an Export (Recovery) of a database, you need to do the following
1. Use **Connect-RSC** to Connect to RSC
1. Use **Get-RscCluster** to get the cluster id that the SQL Server is registered. This is akin to the datacenter that the SQL Server is located in. While you can run the next command without the cluster ID, you may get more records back than expected. This is because your SQL Server Host could exist on 2 different cluster. Once as a replicated source and once as the replicated target. 
1. Use **Get-RscMssqlInstance** to get the instance id of the SQL Server based on the name of the instance and the cluster where it is located. 
1. Use **Get-RscMssqlDatabase** to get the database ID based on the database name and the SQL Server Instance. 
1. Use **Get-RscMssqlDatabaseRecoveryPoint** to get the fully formatted date that corresponds to the recovery point selected.
1. Use **Get-RscMssqlDatabaseFiles** to get the list of files from the backup. 
1. Use **Get-RscMssqlInstance** to get the instance id of the Target SQL Server based on the name of the instance and the cluster where it is located. 
1. Use **New-RscMssqlExport** to start an in place restore of the database to the point in time provided. 
1. Use **Get-RscRequest** to track the request and know when it completes. ***DOES NOT EXIST YET***
1. Use **Disconnect-RSC** to disconnect from RSC

In [13]:
$ClusterName = "cl-rbk-edge-1"
$SourceHostName = "cl-mssql-1.tpm.lab"
$DatabaseName = "AdventureWorks2022"
$TargetHostName = "cl-mssql-1.tpm.lab"
$AvailabilityGroupName = "AG1"

## Connect to RSC

In [14]:
Import-Module RubrikSecurityCloud
$ConnectRsc = @{
    Server = op read "op://Rubrik/TPMServiceAccount/hostname"
    ClientId = op read "op://Rubrik/TPMServiceAccount/username"
    ClientSecret = ConvertTo-SecureString (op read "op://Rubrik/TPMServiceAccount/credential") -AsPlainText -Force
}
Connect-RSC @ConnectRsc

ServiceAccountDetails


ClientId,client|c6ab21c8-9933-488a-8b39-6d24ea8d2037
AuthenticationState,AUTHORIZED
SdkSchemaVersion,v20250616-31
ServerVersion,v20250714-24


# Get the Cluster ID

In [15]:
$RscCluster = Get-RscCluster -Name $ClusterName

# Get the Database from an Instance

## Get the Instance Information

In [ ]:
$RscMssqlInstance = Get-RscMssqlInstance -RscCluster $RscCluster -HostName $SourceHostName -Replica:$false -Relic:$false
$RscMssqlInstance

## Get the Database Information

In [ ]:
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlInstance $RscMssqlInstance -Relic:$false -Replica:$false
$RscMssqlDatabase

# Get the Database from an Availability Group

## Get the Availability Group 

In [16]:
$RscMssqlAvailabilityGroup = Get-RscMssqlAvailabilityGroup -AvailabilityGroupName $AvailabilityGroupName -RscCluster $RscCluster -Replica:$false -Relic:$false
$RscMssqlAvailabilityGroup

RubrikSecurityCloud.Types.MssqlAvailabilityGroup AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType <null> SlaAssignment <null> ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid <null> Description <null> Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived <null> IsDefault <null> IsReadOnly <null> IsRetentionLockedSla <null> Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion <null> UiColor <null> Version <null> AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CopyOnly <null> HasLogConfigFromSla <null> HostLogRetention <null> Id 92347e4d-5b18-5102-be9f-a33b7c1597c7 LogBackupFrequencyInSeconds <null> LogBackupRetentionInHours <null> Name AG1 NumWorkloadDescendants <null> ReplicatedObjectCount <null> SlaPauseStatus <null> AllOrgs <null> AllTags <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> MetricTimeSeriesNew key value timeRange <null> unit <null> SnappableConnection key value first <null> after <null> last <null> before <null> filter <null> CrossAccountReplicatedObjectInfos <null> DescendantConnection <null> EffectiveSlaSourceObject <null> Instances <null> LatestUserNote <null> LogicalChildConnection <null> LogicalPath <null> ObjectPauseStatus <null> PendingObjectDeletionStatus <null> PhysicalPath <null> PrimaryClusterLocation RubrikSecurityCloud.Type

### Get the Database Information

In [17]:
$RscMssqlDatabase = Get-RscMssqlDatabase -Name $DatabaseName -RscMssqlAvailabilityGroup $RscMssqlAvailabilityGroup -Replica:$false -Relic:$false
$RscMssqlDatabase

RubrikSecurityCloud.Types.MssqlDatabase AuthorizedOperations <null> CdmPendingObjectPauseAssignment <null> ObjectType MSSQL SlaAssignment DERIVED ConfiguredSlaDomain <null> EffectiveRetentionSlaDomain <null> EffectiveSlaDomain RubrikSecurityCloud.Types.GlobalSlaReply ObjectTypes <null> RetentionLockMode <null> ClusterUuid 00000000-0000-0000-0000-000000000000 Description Id 99e55090-f8ab-4f5e-8956-ccf4bbfae442 IsArchived False IsDefault False IsReadOnly False IsRetentionLockedSla False Name CL-MSSQL OwnerOrgName <null> ProtectedObjectCount <null> SnapshotScheduleLastUpdatedAt <null> StateVersion 3 UiColor <null> Version AllOrgsHavingAccess <null> AllOrgsWithAccess <null> ArchivalLocationsUpgradeInfo <null> ArchivalSpec <null> ArchivalSpecs <null> BackupWindows <null> BaseFrequency <null> ClusterToSyncStatusMap <null> FirstFullBackupWindows <null> LocalRetentionLimit <null> LogConfig <null> ObjectSpecificConfigs <null> OwnerOrg <null> PausedClustersInfo <null> ReplicationSpec <null> ReplicationSpecsV2 <null> SnapshotSchedule <null> SourceClusters <null> UpgradeInfo <null> PendingSla <null> ReplicatedObjects <null> CdmId 97b0d65b-8d10-4c6b-8bb8-1e6beb1f7d1c CdmLink <null> CdmOnDemandSnapshotCount <null> CopyOnly False DagId 3f2074b2-e886-5fa7-990d-83ccc1a9d093 HasLogConfigFromSla True HasPermissions True HostLogRetention -1 Id 2cb8ad8b-ecb6-51c6-b22c-37ef22da6085 IsInAvailabilityGroup True IsLogShippingSecondary False IsMount False IsOnline True IsRelic False LogBackupFrequencyInSeconds -2 LogBackupRetentionInHours -3 Name AdventureWorks2022 NumWorkloadDescendants 0 OnDemandSnapshotCount <null> PostBackupScript PreBackupScript RecoveryModel <null> ReplicatedObjectCount <null> SlaPauseStatus False UnprotectableReasons <null> Version 16.0.1140.6 AllOrgs <null> AllTags <null> CdmGroupedSnapshots <null> CdmNewestSnapshot <null> CdmOldestSnapshot <null> CdmSnapshots <null> Cluster RubrikSecurityCloud.Types.Cluster CyberEventLockdownMode <null> EosStatus <null> LicensedProducts <null> PauseStatus <null> ProductType <null> RegisteredMode <null> Status <null> StatusFromDb <null> SubStatus <null> SystemStatus <null> Type <null> CdmRbacMigrationStatus <null> ConnectivityLastUpdated <null> DefaultAddress <null> DefaultPort <null> EncryptionEnabled <null> EosDate <null> EstimatedRunway <null> Id 10355ecf-8233-45bb-88d2-5db926d96b4c IsAirGapped <null> IsClusterRemovalTprEnabled <null> IsHealthy <null> IsTprEnabled <null> LastConnectionTime <null> Name cl-rbk-edge-1 NoSqlWorkloadCount <null> PassesConnectivityCheck <null> RawAddress <null> RegistrationTime <null> SnapshotCount <null> SystemStatusMessage <null> Timezone <null> Version <null> ActivitySeriesConnection <null> AllOrgs <null> AuthorizedOperations <null> CcprovisionInfo <null> CdmClusterNodeDetails <null> CdmNotificationSettings <null> CdmUpgradeInfo <null> CloudInfo <null> ClusterDiskConnection <null> ClusterNodeConnection <null> ClusterNodeStats <null> ConfigProtectionInfo <null> CyberEventLockdownSupportCaseDetails <null> DatagovAutoEnablePolicyConfig <null> DatagovPreviewerConfig <null> GeoLocation <null> GlobalManagerConnectivityStatus <null> IpmiInfo <null> LambdaConfig <null> LambdaFeatureHistory <null> MetadataPullScheduler <null> Metric <null> MetricTimeSeries <null> MetricTimeSeriesNew <null> ReplicationSources <null> ReplicationTargets <null> RubrikSyncStatus <null> SnappableConnection <null> State <null> SystemStatusAffectedNodes <null> WebServerCertificate <null> Vars RubrikSecurityCloud.Types.Cluster+InlineVars ActivitySeriesConnection key value first <null> after <null> last <null> before <null> sortBy <null> filters <null> ClusterDiskConnection key value first <null> after <null> last <null> before <null> filter <null> ClusterNodeConnection key value first <null> after <null> last <null> before <null> filter <null> sortBy <null> sortOrder <null> ClusterNodeStats key value cdmClusterNodeID <null> timeRange <null> MetricTimeSeries key value timeRange <null> unit <null> 

# Get the fully formatted recovery point

In [18]:
# $RecoveryDateTime = Get-RscMssqlDatabaseRecoveryPoint -RscMssqlDatabase $RscMssqlDatabase -Latest
$RecoveryDateTime = Get-RscMssqlDatabaseRecoveryPoint -RscMssqlDatabase $RscMssqlDatabase -LastFull
$RecoveryDateTime

2025-07-24T19:30:53.000Z


## Get the Target Instance Information

In [19]:
$TargetMssqlInstance = Get-RscMssqlInstance -RscCluster $RscCluster -HostName $TargetHostName

## Export using the Simple Method

In [ ]:
$NewRscMssqlExport = @{
    RscMssqlDatabase = $RscMssqlDatabase
    RecoveryDateTime = $RecoveryDateTime
    TargetMssqlInstance = $TargetMssqlInstance
    TargetDatabaseName = "test_simple_method"
    TargetDataPath = "c:\mnt\sqldata\"
    TargeLogPath = "c:\mnt\sqllogs\"
    Overwrite = $true
    FinishRecovery = $true
}
New-RscMssqlExport @NewRscMssqlExport

## Get list of files of the database from the backup

In [20]:
$RscMssqlDatabaseFiles = Get-RscMssqlDatabaseFiles -RscMssqlDatabase $RscMssqlDatabase -RecoveryDateTime $RecoveryDateTime
$RscMssqlDatabaseFiles

FileType,MSSQL_DATABASE_FILE_TYPE_DATA
FileId,1
LogicalName,AdventureWorks2022
OriginalName,AdventureWorks2022.mdf
OriginalPath,C:\mnt\sqldata


FileType,MSSQL_DATABASE_FILE_TYPE_LOG
FileId,2
LogicalName,AdventureWorks2022_log
OriginalName,AdventureWorks2022_log.ldf
OriginalPath,C:\mnt\sqllogs


## Create hash table of files

In [21]:
$TargetFilePaths = @()
foreach($File in $RscMssqlDatabaseFiles){
    If ($File.FileType -eq 'MSSQL_DATABASE_FILE_TYPE_DATA'){
        $TargetDataPath = @{
            exportPath = $File.OriginalPath
            logicalName = $File.LogicalName
            newFilename = "Lumnah_Test.mdf"
        }
        $TargetFilePaths += $TargetDataPath
    }

    If ($File.FileType -eq 'MSSQL_DATABASE_FILE_TYPE_LOG'){
        $TargetLogPath = @{
            exportPath = $File.OriginalPath
            logicalName = $File.LogicalName
            newFilename = "Lumnah_Test_log.ldf"
        } 
        $TargetFilePaths += $TargetLogPath
    }
}
$TargetFilePaths

key,value
newFilename,Lumnah_Test.mdf
logicalName,AdventureWorks2022
exportPath,C:\mnt\sqldata


key,value
newFilename,Lumnah_Test_log.ldf
logicalName,AdventureWorks2022_log
exportPath,C:\mnt\sqllogs


## Export using the Advanced Method

In [22]:
$NewRscMssqlExport = @{
    RscMssqlDatabase = $RscMssqlDatabase
    RecoveryDateTime = $RecoveryDateTime
    TargetMssqlInstance = $TargetMssqlInstance
    TargetDatabaseName = "test_advanced_method"
    TargetFilePaths = $TargetFilePaths
    Overwrite = $true
    FinishRecovery = $true
}
New-RscMssqlExport @NewRscMssqlExport

EndTime,<null>
Id,RESTORE_MSSQL_DB_06a247df-b0fb-413e-990c-9b9ace002bf1_caf1290d-01eb-460b-b43c-0c1da0d059a0:::0
NodeId,<null>
Progress,<null>
Result,<null>
StartTime,<null>
Status,QUEUED
Error,<null>
Links,<null>


## Track the status of the restore

---

# Disconnect from RSC
It is important to have each script you create end by disconnecting from RSC. If you do not, then you will run into issues with token authentication exhaustion. RSC allows a Service Account to have 10 concurrent open connections to RSC. Additional requests will be denied until existing connections are closed. 

You can easily disconnect your connection by running the below statement. 

In [ ]:
Disconnect-Rsc